# 환경변수, 시크릿, 헬스체크, 로그

Docker Desktop의 실습 전용 이미지와 스택만 사용해 결과를 확인합니다.

## 실행 환경 준비

마지막 실습도 같은 전용 프로젝트만 사용합니다.

### 이 준비 셀이 하는 일

이 준비 셀은 현재 폴더와 부모 폴더에서 `Dockerfile.baseline`이 있는 Ch32 실습 폴더를 찾습니다. `run(*args, check=True)`은 모든 Docker 명령을 그 폴더에서 실행하고 표준 출력·오류와 종료 코드를 `CompletedProcess`로 반환합니다. 실패 자체를 관찰하는 셀만 `check=False`를 전달하며, 마지막에는 Docker Engine이 응답하는지 확인합니다.

In [1]:
# Docker 명령을 일정한 폴더에서 실행하고 실패도 결과로 받습니다.
from pathlib import Path
import json, subprocess, time, urllib.request
# 챕터 폴더와 notebooks 폴더 어디에서 시작해도 제공 자료를 찾습니다.
candidates = [parent / "labs/part7/ch32-docker" for parent in (Path.cwd(), *Path.cwd().parents)]
candidates += [Path.cwd(), Path.cwd().parent]
LAB = next(path for path in candidates if (path / "Dockerfile.baseline").is_file())
def run(*args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    return subprocess.run(args, cwd=LAB, text=True, capture_output=True, check=check)
version = run("docker", "version", "--format", "{{.Server.Version}}")
print("Docker Engine", version.stdout.strip())
assert version.stdout.strip()
run("docker", "compose", "-p", "ch32-course-stack", "down", "--volumes", check=False)

Docker Engine 29.4.1


CompletedProcess(args=('docker', 'compose', '-p', 'ch32-course-stack', 'down', '--volumes'), returncode=0, stdout='', stderr='time="2026-09-11T11:07:07+09:00" level=warning msg="Warning: No resource found to remove for project \\"ch32-course-stack\\"."\n')

## 환경변수와 시크릿 위치

Compose 설정에는 시크릿 값 대신 컨테이너 파일 경로가 들어갑니다.

### 바로 다음 코드를 읽는 방법

환경변수는 일반 설정을 전달하기 편하지만 프로세스 정보와 디버그 출력에 노출될 수 있습니다. Compose secret은 허용된 서비스에 파일로 마운트되며, 예제의 `DB_PASSWORD_FILE`은 `/run/secrets/db_password` 경로를 가리킵니다. 애플리케이션은 경로 문자열이 아니라 그 파일의 내용을 읽어 인증에 사용합니다.

In [2]:
# 정규화한 설정에서 파일 기반 시크릿 연결을 확인합니다.
config = json.loads(run("docker", "compose", "-p", "ch32-course-stack", "config", "--format", "json").stdout)
app_env = config["services"]["app"]["environment"]
print("DB_PASSWORD_FILE", app_env["DB_PASSWORD_FILE"])
assert app_env["DB_PASSWORD_FILE"] == "/run/secrets/db_password"

DB_PASSWORD_FILE /run/secrets/db_password


## 실패 재현

잘못된 DB 호스트를 준 단독 앱은 살아 있지만 준비 상태가 아닙니다.

In [3]:
# DB_HOST를 고의로 틀리고 liveness 200과 readiness 503을 함께 확인합니다.
started = run("docker", "run", "-d", "--rm", "--name", "ch32-unready", "-p", "127.0.0.1:18333:8000", "-e", "DB_HOST=missing", "ch32-optimized:local")
time.sleep(2)
health = run("curl", "-sS", "-o", "/dev/null", "-w", "%{http_code}", "http://127.0.0.1:18333/health", check=False)
ready = run("curl", "-sS", "-o", "/dev/null", "-w", "%{http_code}", "http://127.0.0.1:18333/ready", check=False)
print("health status", health.stdout, "ready status", ready.stdout)
run("docker", "stop", "ch32-unready")
assert health.stdout == "200" and ready.stdout == "503"

health status 200 ready status 503


## 수정

Compose가 DB 준비를 확인한 다음 app을 시작합니다.

In [4]:
# 정상 스택에서는 Compose healthcheck와 앱 readiness를 따로 확인합니다.
run("docker", "compose", "-p", "ch32-course-stack", "up", "--build", "--wait")
raw_ps = run("docker", "compose", "-p", "ch32-course-stack", "ps", "--format", "json").stdout
ps = [json.loads(line) for line in raw_ps.splitlines() if line]
states = {item["Service"]: item["Health"] for item in ps}
ready = run("curl", "-sS", "-o", "/dev/null", "-w", "%{http_code}", "http://127.0.0.1:18332/ready")
print(states, "ready status", ready.stdout)
assert states == {"app": "healthy", "db": "healthy"} and ready.stdout == "200"

{'app': 'healthy', 'db': 'healthy'} ready status 200


## 로그 확인과 정리

최근 로그에서 요청과 준비 메시지를 확인한 뒤 전용 볼륨까지 삭제합니다.

### 이 단계에서 구분할 범위

liveness는 프로세스가 응답할 수 있는지, readiness는 현재 실제 트래픽을 받을 준비가 됐는지 확인합니다. 데이터베이스 주소를 틀린 컨테이너가 `/health` 200과 `/ready` 503을 함께 반환하는 이유입니다. 이번 실습은 두 HTTP 상태와 Compose의 health 표시까지만 확인합니다. 자동 재시작과 트래픽 제외는 오케스트레이터가 별도 정책을 연결할 때 동작하며 healthcheck 자체가 보장하지 않습니다.

In [5]:
# HTTP 요청을 만든 뒤 최근 로그만 읽습니다.
urllib.request.urlopen("http://127.0.0.1:18332/vector-count").read()
logs = run("docker", "compose", "-p", "ch32-course-stack", "logs", "--no-color", "--tail", "20", "app")
print("has request log", "vector-count" in logs.stdout, "secret exposed", "replace-this" in logs.stdout)
assert "vector-count" in logs.stdout and "replace-this" not in logs.stdout
run("docker", "compose", "-p", "ch32-course-stack", "down", "--volumes")

has request log True secret exposed False


CompletedProcess(args=('docker', 'compose', '-p', 'ch32-course-stack', 'down', '--volumes'), returncode=0, stdout='', stderr=' Container ch32-course-stack-app-1 Stopping \n Container ch32-course-stack-app-1 Stopped \n Container ch32-course-stack-app-1 Removing \n Container ch32-course-stack-app-1 Removed \n Container ch32-course-stack-db-1 Stopping \n Container ch32-course-stack-db-1 Stopped \n Container ch32-course-stack-db-1 Removing \n Container ch32-course-stack-db-1 Removed \n Volume ch32-course-stack_pgdata Removing \n Network ch32-course-stack_default Removing \n Volume ch32-course-stack_pgdata Removed \n Network ch32-course-stack_default Removed \n')